In [12]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from shapely.geometry import shape, Polygon, MultiPolygon, GeometryCollection, box
from shapely.ops import unary_union
from shapely.validation import make_valid
from sahi.predict import get_sliced_prediction
from sahi.models.ultralytics import UltralyticsDetectionModel
from glob import glob
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import io
import base64
import gc

# Last

In [13]:

# =========================
# FUNCIONES BASE
# =========================


def segmentation_to_polygon(poly):
    if isinstance(poly[0], (list, tuple)):
        return Polygon(poly)
    else:
        coords = [(poly[i], poly[i+1]) for i in range(0, len(poly), 2)]
        return Polygon(coords)

def extract_polygons(geom):
    polygons = []
    if geom.is_empty:
        return polygons
    if isinstance(geom, Polygon):
        polygons.append(geom)
    elif isinstance(geom, MultiPolygon):
        polygons.extend(list(geom.geoms))
    elif isinstance(geom, GeometryCollection):
        for g in geom.geoms:
            if isinstance(g, Polygon):
                polygons.append(g)
            elif isinstance(g, MultiPolygon):
                polygons.extend(list(g.geoms))
    return polygons

def create_tile_grid(img_width, img_height, tile_size, overlap_ratio):
    step = int(tile_size * (1 - overlap_ratio))
    tiles = []
    tile_id = 0
    
    for y in range(0, img_height - tile_size + 1, step):
        for x in range(0, img_width - tile_size + 1, step):
            x_end = min(x + tile_size, img_width)
            y_end = min(y + tile_size, img_height)
            x_start = x_end - tile_size
            y_start = y_end - tile_size
            
            tiles.append({
                'id': tile_id,
                'bbox': (x_start, y_start, x_end, y_end),
                'polygon': box(x_start, y_start, x_end, y_end),
                'row': y // step,
                'col': x // step,
                'center_x': (x_start + x_end) / 2,
                'center_y': (y_start + y_end) / 2
            })
            tile_id += 1
    
    return tiles

def clip_geometry_to_tile(geom, tile_polygon):
    if geom is None or geom.is_empty:
        return Polygon()
    try:
        intersection = geom.intersection(tile_polygon)
        return make_valid(intersection) if not intersection.is_empty else Polygon()
    except Exception:
        return Polygon()

def compute_metrics(pred_geom, gt_geom):
    if pred_geom is None or pred_geom.is_empty:
        inter = 0
        pred_area = 0
    else:
        pred_area = pred_geom.area
        inter = pred_geom.intersection(gt_geom).area if not gt_geom.is_empty else 0

    gt_area = gt_geom.area if not gt_geom.is_empty else 0
    union = pred_area + gt_area - inter

    iou = inter / union if union > 0 else 0
    dice = (2 * inter) / (pred_area + gt_area) if (pred_area + gt_area) > 0 else 0
    precision = inter / pred_area if pred_area > 0 else 0
    recall = inter / gt_area if gt_area > 0 else 0

    return {
        "iou": iou,
        "dice": dice,
        "precision": precision,
        "recall": recall,
        "f1": dice,
        "intersection_area": float(inter),
        "pred_area": float(pred_area),
        "gt_area": float(gt_area),
        "union_area": float(union)
    }

# =========================
# MATRIZ DE CONFUSIÓN (NORMALIZADA POR FILAS - PREDICCIÓN)
# =========================
def compute_confusion_matrix_normalized(pred_by_class, gt_by_class, img_width, img_height):
    """
    Matriz de confusión normalizada con Pixel Accuracy correcto.
    """
    classes = [0, 1, 2]
    total_pixels = img_width * img_height  # Área total de la imagen
    
    # Crear máscaras binarias completas
    # Unión de todas las predicciones (fondo = lo que no es ninguna clase)
    all_pred = unary_union([pred_by_class.get(c, Polygon()) for c in classes])
    
    # Unión de todo el GT
    all_gt = unary_union([gt_by_class.get(c, Polygon()) for c in classes])
    
    # Matriz de confusión (solo áreas de intersección)
    confusion = np.zeros((3, 3), dtype=np.float64)
    
    for i, pred_cls in enumerate(classes):
        pred_mask = pred_by_class.get(pred_cls, Polygon())
        for j, gt_cls in enumerate(classes):
            gt_mask = gt_by_class.get(gt_cls, Polygon())
            if not pred_mask.is_empty and not gt_mask.is_empty:
                inter = pred_mask.intersection(gt_mask).area
                confusion[i, j] = inter
    
    # === CÁLCULO CORRECTO DEL PIXEL ACCURACY ===
    
    # Opción 1: Pixel Accuracy estándar (pixeles correctos / total pixeles)
    # Suma de diagonal = área donde predicción coincide con GT
    correct_pixels = np.trace(confusion)
    pixel_acc_standard = correct_pixels / total_pixels if total_pixels > 0 else 0
    
    # Opción 2: Mean Pixel Accuracy (promedio de accuracy por clase)
    # Para cada clase: pixeles correctos de esa clase / pixeles totales de esa clase en GT
    col_sums = confusion.sum(axis=0)  # Suma por columnas = área GT de cada clase
    pixel_acc_per_class = np.divide(np.diag(confusion), col_sums, 
                                    out=np.zeros(3), 
                                    where=col_sums!=0)
    mean_pixel_acc = pixel_acc_per_class.mean()
    
    # Opción 3: Tu cálculo actual (porcentaje de predicción correcta)
    total_pred_area = confusion.sum()
    pixel_acc_pred_based = correct_pixels / total_pred_area if total_pred_area > 0 else 0
    
    # Normalizar matriz por filas (como tenías)
    row_sums = confusion.sum(axis=1, keepdims=True)
    confusion_norm = np.divide(confusion, row_sums, 
                               out=np.zeros_like(confusion), 
                               where=row_sums!=0)
    
    # DataFrame: Filas=Predicción, Columnas=GT
    df_confusion = pd.DataFrame(
        confusion_norm,
        index=[f'Pred: {class_names[i]}' for i in classes],
        columns=[f'GT: {class_names[i]}' for i in classes]
    )
    return {
        'matrix_raw': confusion,
        'matrix_norm': confusion_norm,
        'pixel_accuracy_standard': pixel_acc_standard,      # Correcto: correctos/total
        'pixel_accuracy_mean': mean_pixel_acc,               # Mean per-class accuracy
        'pixel_accuracy_pred_based': pixel_acc_pred_based,   # Tu cálculo actual
        'total_pixels': total_pixels,
        'correct_pixels': correct_pixels,
        'predicted_area': total_pred_area,
        'gt_area': col_sums.sum(),
        'dataframe': df_confusion
    }



# =========================
# ANÁLISIS POR TILES
# =========================

def compute_tile_metrics_detailed(pred_by_class, gt_by_class, tile_info):
    tile_polygon = tile_info['polygon']
    tile_area = tile_polygon.area
    
    results = {
        'tile_id': tile_info['id'],
        'tile_row': tile_info['row'],
        'tile_col': tile_info['col'],
        'center_x': tile_info['center_x'],
        'center_y': tile_info['center_y'],
        'tile_area': float(tile_area),
        'by_class': {}
    }
    
    all_classes = set(list(pred_by_class.keys()) + list(gt_by_class.keys()))
    
    for cls in sorted(all_classes):
        full_pred = pred_by_class.get(cls, Polygon())
        full_gt = gt_by_class.get(cls, Polygon())
        
        pred_in_tile = clip_geometry_to_tile(full_pred, tile_polygon)
        gt_in_tile = clip_geometry_to_tile(full_gt, tile_polygon)
        
        metrics = compute_metrics(pred_in_tile, gt_in_tile)
        
        gt_coverage = gt_in_tile.area / tile_area
        pred_coverage = pred_in_tile.area / tile_area
        
        results['by_class'][cls] = {
            'class_id': cls,
            'class_name': class_names.get(cls, f"Class_{cls}"),
            'metrics': metrics,
            'gt_coverage': float(gt_coverage),
            'pred_coverage': float(pred_coverage),
            'gt_area': float(gt_in_tile.area),
            'pred_area': float(pred_in_tile.area),
            'is_evaluable': gt_coverage >= MIN_GT_COVERAGE,
            'has_pred': pred_in_tile.area > 0,
            'has_gt': gt_in_tile.area > 0
        }
    
    return results

def aggregate_statistics_by_class(tile_results_list):
    class_stats = defaultdict(lambda: {
        'tiles_evaluable': [],
        'tiles_with_pred': [],
        'tiles_without_pred': [],
        'all_iou': [],
        'all_dice': [],
        'all_precision': [],
        'all_recall': [],
        'coverages': [],
        'positions': []
    })
    
    for tile_data in tile_results_list:
        for cls, cls_data in tile_data['by_class'].items():
            if not cls_data['is_evaluable']:
                continue
            
            stats = class_stats[cls]
            stats['tiles_evaluable'].append(tile_data['tile_id'])
            stats['coverages'].append(cls_data['gt_coverage'])
            stats['positions'].append((tile_data['center_x'], tile_data['center_y']))
            
            metrics = cls_data['metrics']
            
            if cls_data['has_pred']:
                stats['tiles_with_pred'].append(tile_data['tile_id'])
                stats['all_iou'].append(metrics['iou'])
                stats['all_dice'].append(metrics['dice'])
                stats['all_precision'].append(metrics['precision'])
                stats['all_recall'].append(metrics['recall'])
            else:
                stats['tiles_without_pred'].append(tile_data['tile_id'])
                stats['all_iou'].append(0.0)
                stats['all_dice'].append(0.0)
                stats['all_precision'].append(0.0)
                stats['all_recall'].append(0.0)
    
    aggregated = {}
    
    for cls, stats in class_stats.items():
        if not stats['all_iou']:
            continue
        
        iou_array = np.array(stats['all_iou'])
        dice_array = np.array(stats['all_dice'])
        precision_array = np.array(stats['all_precision'])
        recall_array = np.array(stats['all_recall'])
        
        aggregated[cls] = {
            'class_name': class_names.get(cls, f"Class_{cls}"),
            'n_tiles_evaluable': len(stats['tiles_evaluable']),
            'n_tiles_with_pred': len(stats['tiles_with_pred']),
            'n_tiles_without_pred': len(stats['tiles_without_pred']),
            'coverage_mean': float(np.mean(stats['coverages'])),
            'coverage_std': float(np.std(stats['coverages'])),
            'iou': {
                'mean': float(np.mean(iou_array)),
                'std': float(np.std(iou_array)),
                'median': float(np.median(iou_array)),
                'min': float(np.min(iou_array)),
                'max': float(np.max(iou_array)),
                'q25': float(np.percentile(iou_array, 25)),
                'q75': float(np.percentile(iou_array, 75)),
            },
            'dice': {
                'mean': float(np.mean(dice_array)),
                'std': float(np.std(dice_array)),
                'median': float(np.median(dice_array)),
            },
            'precision': {
                'mean': float(np.mean(precision_array)),
                'std': float(np.std(precision_array)),
                'median': float(np.median(precision_array))
            },
            'recall': {
                'mean': float(np.mean(recall_array)),
                'std': float(np.std(recall_array)),
                'median': float(np.median(recall_array))
            },
            'raw_data': {
                'iou': iou_array.tolist(),
                'dice': dice_array.tolist(),
                'precision': precision_array.tolist(),
                'recall': recall_array.tolist(),
                'positions': stats['positions']
            }
        }
    
    return aggregated

def create_summary_table(aggregated_stats):
    rows = []
    for cls in sorted(aggregated_stats.keys()):
        stats = aggregated_stats[cls]
        row = {
            'Clase': stats['class_name'],
            'Tiles Evaluables': stats['n_tiles_evaluable'],
            'Con Predicción (%)': f"{stats['n_tiles_with_pred']/stats['n_tiles_evaluable']*100:.1f}",
            'Sin Predicción (%)': f"{stats['n_tiles_without_pred']/stats['n_tiles_evaluable']*100:.1f}",
            'IoU (media)': f"{stats['iou']['mean']:.4f}",
            'IoU (std)': f"{stats['iou']['std']:.4f}",
            'IoU (mediana)': f"{stats['iou']['median']:.4f}",
            'IoU [min, max]': f"[{stats['iou']['min']:.3f}, {stats['iou']['max']:.3f}]",
            'Dice (media±std)': f"{stats['dice']['mean']:.3f}±{stats['dice']['std']:.3f}",
            'Recall': f"{stats['recall']['mean']:.3f}",
            'Precision': f"{stats['precision']['mean']:.3f}",
            'Cobertura GT (%)': f"{stats['coverage_mean']*100:.1f}±{stats['coverage_std']*100:.1f}"
        }
        rows.append(row)
    
    return pd.DataFrame(rows)

# =========================
# VISUALIZACIÓN
# =========================

def create_overlay_image(image_path, pred_by_class, alpha=0.6, is_pred=True):
    img = Image.open(image_path).convert('RGBA')
    
    max_dim = 2048
    scale = 1.0
    if max(img.size) > max_dim:
        scale = max_dim / max(img.size)
        img = img.resize((int(img.size[0]*scale), int(img.size[1]*scale)), Image.Resampling.LANCZOS)
    
    overlay = Image.new('RGBA', img.size, (0,0,0,0))
    draw = ImageDraw.Draw(overlay)
    
    for cls, geom in pred_by_class.items():
        if geom.is_empty:
            continue
        
        polys = extract_polygons(geom)
        for poly in polys:
            coords = [(x*scale, y*scale) for x,y in poly.exterior.coords]
            color = colors_rgb.get(cls, (128,128,128)) + (int(255*alpha),)
            draw.polygon(coords, fill=color)
            
            if not is_pred:
                draw.polygon(coords, outline=(color[0],color[1],color[2],255), width=2)
    
    result = Image.alpha_composite(img, overlay)
    return result.convert('RGB')

def img_to_base64(img):
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode()

# =========================
# DASHBOARD MODIFICADO (SIN HEATMAP ESPACIAL, CON VIOLIN, RECALL/PRECISION, DICE)
# =========================
def create_complete_dashboard(aggregated_stats, tile_results_list, 
                              image_path, pred_by_class, gt_by_class,
                              img_width, img_height, image_name, output_dir,
                              confusion_result,
                              export_png=False):
    
    print(f"Generando dashboard para {image_name}...")
    
    # Overlays
    print("   🎨 Creando overlays...")
    img_pred = create_overlay_image(image_path, pred_by_class, alpha=0.6, is_pred=True)
    img_gt = create_overlay_image(image_path, gt_by_class, alpha=0.6, is_pred=False)
    
    pred_path = os.path.join(output_dir, f"overlay_pred_{image_name}.png")
    gt_path = os.path.join(output_dir, f"overlay_gt_{image_name}.png")
    img_pred.save(pred_path)
    img_gt.save(gt_path)
    
    pred_b64 = img_to_base64(img_pred)
    gt_b64 = img_to_base64(img_gt)
    
    # Datos
    classes = sorted(aggregated_stats.keys())
    class_labels = [aggregated_stats[c]['class_name'] for c in classes]
    
    # Crear figura 2x3: 6 subplots - SIN matriz de confusión en Plotly
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=(
            'IoU por Clase',
            'Recall vs Precision',
            'Violin Plot: Distribución IoU',
            'Dice por Clase',
            #'F1-Score por Clase',
            'Violin Plot: Distribución Dice'
        ),
        specs=[[{"type": "bar"}, {"type": "bar"}, {"type": "violin"}],
               [{"type": "bar"}, {"type": "bar"}, {"type": "violin"}]],
        vertical_spacing=0.15,
        horizontal_spacing=0.08
    )
    
    # 1. Barras IoU (1,1)
    means_iou = [aggregated_stats[c]['iou']['mean'] for c in classes]
    stds_iou = [aggregated_stats[c]['iou']['std'] for c in classes]
    
    for i, (cls, label, color) in enumerate(zip(classes, class_labels, colors_plotly)):
        fig.add_trace(go.Bar(x=[label], y=[means_iou[i]],
            error_y=dict(type='data', array=[stds_iou[i]], visible=True,
                        color='black', thickness=1.5, width=6),
            marker_color=color, text=f'{means_iou[i]:.3f}',
            textposition='outside', showlegend=False), row=1, col=1)
    
    fig.add_hline(y=0.5, line_dash="dash", line_color="gray", row=1, col=1)
    
    # 2. Recall vs Precision (1,2)
    recall_means = [aggregated_stats[c]['recall']['mean'] for c in classes]
    precision_means = [aggregated_stats[c]['precision']['mean'] for c in classes]
    
    x_pos = np.arange(len(class_labels))
    width = 0.35
    
    for i, (metric, values, color, name) in enumerate([
        ('recall', recall_means, '#9b59b6', 'Recall'),
        ('precision', precision_means, '#f39c12', 'Precision')
    ]):
        fig.add_trace(go.Bar(
            x=[x + (i-0.5)*width for x in x_pos],
            y=values,
            name=name,
            marker_color=color,
            text=[f'{v:.3f}' for v in values],
            textposition='auto',
            width=width,
            showlegend=True
        ), row=1, col=2)
    
    fig.update_xaxes(tickvals=x_pos, ticktext=class_labels, row=1, col=2)
    
    # 3. Violin Plot IoU (1,3) - CON COLORES DE CLASE
    for i, (cls, label, color) in enumerate(zip(classes, class_labels, colors_plotly)):
        data = aggregated_stats[cls]['raw_data']['iou']
        fig.add_trace(go.Violin(
            y=data, 
            name=label, 
            line_color='black',
            fillcolor=color,
            opacity=0.7,
            box_visible=True, 
            meanline_visible=True,
            showlegend=False
        ), row=1, col=3)
    fig.update_yaxes(range=[0, 1], row=1, col=3)
    
    # 4. Dice por Clase (2,1)
    means_dice = [aggregated_stats[c]['dice']['mean'] for c in classes]
    stds_dice = [aggregated_stats[c]['dice']['std'] for c in classes]
    
    for i, (cls, label, color) in enumerate(zip(classes, class_labels, colors_plotly)):
        fig.add_trace(go.Bar(x=[label], y=[means_dice[i]],
            error_y=dict(type='data', array=[stds_dice[i]], visible=True,
                        color='black', thickness=1.5, width=6),
            marker_color=color, text=f'{means_dice[i]:.3f}',
            textposition='outside', showlegend=False), row=2, col=1)
    
    fig.add_hline(y=0.5, line_dash="dash", line_color="gray", row=2, col=1)
    
    # 5. F1-Score por Clase (2,2)
    f1_means = [aggregated_stats[c]['dice']['mean'] for c in classes]
    f1_stds = [aggregated_stats[c]['dice']['std'] for c in classes]
    
    for i, (cls, label, color) in enumerate(zip(classes, class_labels, colors_plotly)):
        fig.add_trace(go.Bar(x=[label], y=[f1_means[i]],
            error_y=dict(type='data', array=[f1_stds[i]], visible=True,
                        color='black', thickness=1.5, width=6),
            marker_color=color, text=f'{f1_means[i]:.3f}',
            textposition='outside', showlegend=False), row=2, col=2)
    
    fig.add_hline(y=0.5, line_dash="dash", line_color="gray", row=2, col=2)
    
    # 6. Violin Plot Dice (2,3) - CON COLORES DE CLASE
    for i, (cls, label, color) in enumerate(zip(classes, class_labels, colors_plotly)):
        data = aggregated_stats[cls]['raw_data']['dice']
        fig.add_trace(go.Violin(
            y=data, 
            name=label, 
            line_color='black',
            fillcolor=color,
            opacity=0.7,
            box_visible=True, 
            meanline_visible=True,
            showlegend=False
        ), row=2, col=3)
    fig.update_yaxes(range=[0, 1], row=2, col=3)
    
    # Layout general
    miou_global = np.mean([aggregated_stats[c]['iou']['mean'] for c in classes])
    pixel_acc = confusion_result['pixel_accuracy_standard'] * 100
    
    fig.update_layout(
        title=dict(text=f'<b>{image_name}</b><br>'
                       f'mIoU: {miou_global:.3f} | Pixel Accuracy Std: {pixel_acc:.1f}% | '
                       f'Tiles: {len([t for t in tile_results_list if any(t["by_class"][c]["is_evaluable"] for c in classes)])}/{len(tile_results_list)}',
                  font_size=14, x=0.5),
        height=1000,
        width=1600,
        template='plotly_white',
        barmode='group',
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.25)
    )
    
    fig.update_yaxes(range=[0, 1], row=1, col=1)
    fig.update_yaxes(range=[0, 1], row=1, col=2)
    fig.update_yaxes(range=[0, 1], row=2, col=1)
    fig.update_yaxes(range=[0, 1], row=2, col=2)
    
    # HTML - MATRIZ DE CONFUSIÓN MANTENIDA EN HTML (solo eliminada de Plotly)
    dashboard_path = os.path.join(output_dir, f"dashboard_complete_{image_name}.html")
    
    # Tabla HTML de confusión (valores normalizados)
    df_conf_html = confusion_result['dataframe'].to_html(
        classes='confusion-table',
        float_format=lambda x: f'{x:.3f}'
    )
    
    html = f"""<!DOCTYPE html>
<html>
<head>
    <title>{image_name}</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 20px; background: #f5f5f5; }}
        .container {{ max-width: 1500px; margin: 0 auto; background: white; padding: 20px; border-radius: 10px; }}
        h1 {{ color: #2c3e50; border-bottom: 3px solid #3498db; padding-bottom: 10px; }}
        h2 {{ color: #34495e; margin-top: 30px; }}
        .images {{ display: flex; gap: 20px; margin: 20px 0; }}
        .img-card {{ flex: 1; background: #fafafa; padding: 15px; border-radius: 8px; }}
        .img-card img {{ width: 100%; border-radius: 4px; border: 1px solid #ddd; }}
        .legend {{ display: flex; gap: 20px; margin: 10px 0; font-size: 14px; }}
        .legend-item {{ display: flex; align-items: center; gap: 5px; }}
        .color-box {{ width: 20px; height: 20px; border-radius: 3px; }}
        .chart {{ margin: 20px 0; border: 1px solid #e0e0e0; border-radius: 8px; }}
        .confusion-table {{ border-collapse: collapse; margin: 15px 0; font-size: 14px; width: 100%; max-width: 600px; }}
        .confusion-table th, .confusion-table td {{ border: 2px solid #333; padding: 12px; text-align: center; }}
        .confusion-table th {{ background-color: #34495e; color: white; font-weight: bold; }}
        .confusion-table td {{ background-color: #ecf0f1; }}
        .metrics-box {{ display: flex; gap: 40px; margin: 20px 0; padding: 20px; background: #e8f4f8; border-radius: 8px; justify-content: center; }}
        .metric {{ text-align: center; }}
        .metric-value {{ font-size: 32px; font-weight: bold; color: #2c3e50; }}
        .metric-label {{ font-size: 14px; color: #666; margin-top: 5px; }}
        .note {{ color: #666; font-size: 13px; margin-top: 10px; font-style: italic; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>🔬 {image_name}</h1>
        
        <div class="metrics-box">
            <div class="metric">
                <div class="metric-value">{miou_global:.3f}</div>
                <div class="metric-label">mIoU</div>
            </div>
            <div class="metric">
                <div class="metric-value">{pixel_acc:.1f}%</div>
                <div class="metric-label">Pixel Accuracy</div>
            </div>
            <div class="metric">
                <div class="metric-value">{sum(aggregated_stats[c]['n_tiles_evaluable'] for c in classes)}</div>
                <div class="metric-label">Tiles Evaluables</div>
            </div>
        </div>
        
        <h2>Segmentaciones</h2>
        <div class="images">
            <div class="img-card">
                <h3>Predicción</h3>
                <img src="data:image/png;base64,{pred_b64}">
                <div class="legend">
                    <div class="legend-item"><div class="color-box" style="background: #e74c3c;"></div>Decidua</div>
                    <div class="legend-item"><div class="color-box" style="background: #2ecc71;"></div>Labyrinth</div>
                    <div class="legend-item"><div class="color-box" style="background: #3498db;"></div>J.Zone</div>
                </div>
            </div>
            <div class="img-card">
                <h3>Ground Truth</h3>
                <img src="data:image/png;base64,{gt_b64}">
                <div class="legend">
                    <div class="legend-item"><div class="color-box" style="background: #e74c3c;"></div>Decidua</div>
                    <div class="legend-item"><div class="color-box" style="background: #2ecc71;"></div>Labyrinth</div>
                    <div class="legend-item"><div class="color-box" style="background: #3498db;"></div>J.Zone</div>
                </div>
            </div>
        </div>
        
        <h2>Métricas</h2>
        <div class="chart" id="chart"></div>
        
        <h3>Matriz de Confusión Normalizada (Filas=Predicción, Columnas=GT)</h3>
        {df_conf_html}
        <p class="note">
            Valores normalizados por fila [0,1]. Verde=1 (predicción coincide perfectamente con GT), Rojo=0 (no coincide).
            La diagonal indica qué proporción de cada clase predicha realmente pertenece a esa clase según GT.
        </p>
    </div>
    <script>
        Plotly.newPlot('chart', {fig.to_json()}.data, {fig.to_json()}.layout, {{responsive: true}});
    </script>
</body>
</html>"""
    
    with open(dashboard_path, 'w', encoding='utf-8') as f:
        f.write(html)
    
    print(f"   ✅ Dashboard: {dashboard_path}")
    
    if export_png:
        try:
            fig.write_image(os.path.join(output_dir, f"dashboard_{image_name}.png"), 
                          scale=2, width=1400, height=800)
            print(f"      📸 PNG exportado")
        except Exception as e:
            print(f"      ⚠️ PNG no disponible: {e}")
    
    return {
        'dashboard': dashboard_path,
        'overlay_pred': pred_path,
        'overlay_gt': gt_path
    }





In [ ]:
# CONFIG PIPELINE
# =========================
models_paths = glob('/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/notebooks/runs/segment/YOLO-lci1/*/weights/best.pt')
images_paths_test = glob('/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/wsi_splitted/test/*.tif')
for model_path in models_paths:

    model_name = model_path.split('/')[-3]

    RAW_OUTPUT = '/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/models/inference_1903/'
    OUTPUT_DIR = os.path.join(RAW_OUTPUT, model_name)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    slice_model = int(model_name.split('-')[1].split('r')[-1])
    slice_size = slice_model
    overlap_ratio = .25
    confidence_threshold = 0.25
    MIN_GT_COVERAGE = 0.05

    name_to_id = {"Decidual": 0, "Decidua": 0, "Labyrinth": 1, "Junctional Zone": 2}
    class_names = {0: "Decidua", 1: "Labyrinth", 2: "Junctional Zone"}
    colors_rgb = {0: (231, 76, 60), 1: (46, 204, 113), 2: (52, 152, 219)}
    colors_plotly = ['#e74c3c', '#2ecc71', '#3498db']

    print(f"Modelo: {model_name}")
    print(f"Slice size: {slice_size} | Overlap: {overlap_ratio*100:.0f}% | Conf. threshold: {confidence_threshold} | Min GT coverage: {MIN_GT_COVERAGE*100:.0f}%")



    # =========================
    # PIPELINE PRINCIPAL
    # =========================

    

    print("Cargando modelo...")
    model = UltralyticsDetectionModel(
        model_path=model_path,
        confidence_threshold=confidence_threshold,
        load_at_init=True
    )

    for image_path in images_paths_test:
        image_name = os.path.basename(image_path).split('.')[0]
        annot_path = f"/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/annotations_resized/{image_name}.geojson"
        OUTPUT_DIR_IMAGE = os.path.join(OUTPUT_DIR, image_name)
        os.makedirs(OUTPUT_DIR_IMAGE, exist_ok=True)
        
        print(f"\n{'='*60}")
        print(f"PROCESANDO: {image_name}")
        print(f"{'='*60}")
        
        if not os.path.exists(annot_path):
            print(f"⚠️  GT no encontrado, saltando...")
            continue
        
        # Cargar imagen
        img = Image.open(image_path)
        img_width, img_height = img.size
        print(f"Tamaño: {img_width}x{img_height}")
        
        # Crear grid
        tiles = create_tile_grid(img_width, img_height, slice_size, overlap_ratio)
        print(f"Tiles: {len(tiles)}")
        
        # Inferencia SAHI
        print("SAHI inference...")
        result = get_sliced_prediction(
            image_path, model,
            slice_height=slice_size, slice_width=slice_size,
            overlap_height_ratio=overlap_ratio, overlap_width_ratio=overlap_ratio
        )
        
        # Procesar predicciones
        pred_by_class_with_scores = {}
        for pred in result.object_prediction_list:
            if pred.mask and hasattr(pred.mask, "segmentation"):
                class_id = pred.category.id
                score = pred.score.value if hasattr(pred.score, 'value') else float(pred.score)
                
                for poly in pred.mask.segmentation:
                    if len(poly) >= 6:
                        raw_poly = segmentation_to_polygon(poly)
                        valid_geom = make_valid(raw_poly)
                        polygons = extract_polygons(valid_geom)
                        
                        for shapely_poly in polygons:
                            if shapely_poly.area > 0:
                                pred_by_class_with_scores.setdefault(class_id, []).append({
                                    'geometry': shapely_poly, 'score': score
                                })
        
        pred_by_class = {cls: unary_union([p['geometry'] for p in preds]) 
                        for cls, preds in pred_by_class_with_scores.items()}
        
        # Cargar GT
        gt_by_class = {}
        with open(annot_path) as f:
            data = json.load(f)
        
        for feature in data.get("features", []):
            geom = make_valid(shape(feature["geometry"]))
            class_name = feature["properties"].get("name")
            if class_name in name_to_id:
                cls_id = name_to_id[class_name]
                gt_by_class.setdefault(cls_id, []).append(geom)
        
        gt_by_class = {cls: unary_union(geoms) for cls, geoms in gt_by_class.items()}
        
        # Calcular matriz de confusión normalizada
        print("Calculando matriz de confusión...")
        confusion_result = compute_confusion_matrix_normalized(pred_by_class, gt_by_class,img_width, img_height)
        print(f"   Pixel Accuracy: {confusion_result['pixel_accuracy_standard']*100:.2f}%")
        print("   Matriz normalizada (Pred→GT):")
        print(confusion_result['dataframe'].round(3).to_string())
        
        # Métricas por tile
        print(f"Calculando métricas por tile...")
        tile_results = [compute_tile_metrics_detailed(pred_by_class, gt_by_class, t) for t in tiles]
        
        # Agregar estadísticas
        aggregated = aggregate_statistics_by_class(tile_results)
        
        # Mostrar resumen
        print(f"\n{'='*60}")
        print("RESULTADOS")
        print(f"{'='*60}")
        
        for cls in sorted(aggregated.keys()):
            s = aggregated[cls]
            print(f"\n📊 {s['class_name']}")
            print(f"   Tiles evaluables: {s['n_tiles_evaluable']}")
            print(f"   IoU: {s['iou']['mean']:.4f} ± {s['iou']['std']:.4f}")
            print(f"   Dice: {s['dice']['mean']:.4f} ± {s['dice']['std']:.4f}")
            print(f"   Recall: {s['recall']['mean']:.4f} | Precision: {s['precision']['mean']:.4f}")
        
        # Tabla y CSV
        df = create_summary_table(aggregated)
        print(f"\nTABLA RESUMEN:")
        print(df.to_string())
        
        csv_path = os.path.join(OUTPUT_DIR_IMAGE, f"summary_{image_name}.csv")
        df.to_csv(csv_path, index=False)
        print(f"\n💾 CSV: {csv_path}")
        
        # JSON
        json_path = os.path.join(OUTPUT_DIR_IMAGE, f"stats_{image_name}.json")
        with open(json_path, 'w') as f:
            export_data = {
                'image': image_name,
                'dimensions': (img_width, img_height),
                'per_class': {k: {kk: vv for kk, vv in v.items() if kk != 'raw_data'} 
                            for k, v in aggregated.items()},
                'confusion_matrix': {
                    'matrix_norm': confusion_result['matrix_norm'].tolist(),
                    'matrix_raw': confusion_result['matrix_raw'].tolist(),
                    'pixel_accuracy_standard': float(confusion_result['pixel_accuracy_standard']),
                    'pixel_accuracy_mean': float(confusion_result['pixel_accuracy_mean']),              
                    'pixel_accuracy_pred_based': float(confusion_result['pixel_accuracy_pred_based']),   
                }
            }
            json.dump(export_data, f, indent=2)
        print(f"💾 JSON: {json_path}")
        
        # Dashboard
        print("Generando dashboard...")
        dashboard_files = create_complete_dashboard(
            aggregated_stats=aggregated,
            tile_results_list=tile_results,
            image_path=image_path,
            pred_by_class=pred_by_class,
            gt_by_class=gt_by_class,
            img_width=img_width,
            img_height=img_height,
            image_name=image_name,
            output_dir=OUTPUT_DIR_IMAGE,
            confusion_result=confusion_result,
            export_png=False
        )
        
        # Liberar memoria
        img.close()
        del result
        gc.collect()

    print(f"\n{'='*60}")
    print("PROCESO COMPLETADO")
    print(f"{'='*60}")

Modelo: yolo26x-256r256-ovlp50-100ep
Slice size: 256 | Overlap: 25% | Conf. threshold: 0.25 | Min GT coverage: 5%
Cargando modelo...

PROCESANDO: slide_021
Tamaño: 9406x5429
Tiles: 1296
SAHI inference...
Performing prediction on 1372 slices.


KeyboardInterrupt: 

## Visualización JSON

In [15]:
import json
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path

def load_all_models_results(base_output_dir):
    """
    Carga todos los resultados JSON de todos los modelos
    base_output_dir: directorio donde están las carpetas de cada modelo
    """
    all_results = []
    
    # Buscar todos los archivos stats_*.json recursivamente
    json_files = glob.glob(os.path.join(base_output_dir, "**/stats_*.json"), recursive=True)
    
    for json_path in json_files:
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        # Extraer nombre del modelo de la ruta
        parts = Path(json_path).parts
        # Asumiendo estructura: .../model_name/image_name/stats_image_name.json
        model_name = parts[-3] if len(parts) >= 3 else "unknown"
        image_name = data.get('image', parts[-2])
        
        # Extraer métricas por clase
        for class_id, class_data in data.get('per_class', {}).items():
            row = {
                'model': model_name,
                'image': image_name,
                'class_id': int(class_id),
                'class_name': class_data['class_name'],
                'iou_mean': class_data['iou']['mean'],
                'iou_std': class_data['iou']['std'],
                'iou_median': class_data['iou']['median'],
                'dice_mean': class_data['dice']['mean'],
                'precision_mean': class_data['precision']['mean'],
                'recall_mean': class_data['recall']['mean'],
                'n_tiles_evaluable': class_data['n_tiles_evaluable'],
                'coverage_mean': class_data['coverage_mean'],
                'pixel_accuracy': data.get('confusion_matrix', {}).get('pixel_accuracy_standard', 0),
                'pixel_accuracy_mean': data.get('confusion_matrix', {}).get('pixel_accuracy_mean', 0),
                'pixel_accuracy_pred_based': data.get('confusion_matrix', {}).get('pixel_accuracy_pred_based', 0)
            }
            all_results.append(row)
    
    return pd.DataFrame(all_results)

# Cargar todos los resultados
df = load_all_models_results('/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/models/inference_1903/')

In [16]:
df

,model,image,class_id,class_name,iou_mean,iou_std,iou_median,dice_mean,precision_mean,recall_mean,n_tiles_evaluable,coverage_mean,pixel_accuracy,pixel_accuracy_mean,pixel_accuracy_pred_based
0,yolo26x-256r256-ovlp50-100ep,slide_010,0,Decidua,0.839465,0.190772,0.903328,0.897461,0.921504,0.904443,157,0.619371,0.258108,0.899277,0.938311
1,yolo26x-256r256-ovlp50-100ep,slide_010,2,Junctional Zone,0.711795,0.240297,0.781030,0.803518,0.831153,0.861067,119,0.441911,0.258108,0.899277,0.938311
2,yolo26x-256r256-ovlp50-100ep,slide_010,1,Labyrinth,0.901462,0.166065,0.998123,0.938201,0.917137,0.979020,322,0.817772,0.258108,0.899277,0.938311
3,yolo26x-256r256-ovlp50-100ep,slide_056,0,Decidua,0.648589,0.340786,0.779502,0.720643,0.801006,0.727739,481,0.679333,0.369973,0.818347,0.850108
4,yolo26x-256r256-ovlp50-100ep,slide_056,2,Junctional Zone,0.680895,0.305065,0.771460,0.764161,0.768880,0.894095,483,0.709658,0.369973,0.818347,0.850108
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,yolo26l-1280r640-1,slide_002,2,Junctional Zone,0.788808,0.156917,0.820785,0.871894,0.857350,0.924284,62,0.429066,0.394319,0.892934,0.931113
146,yolo26l-1280r640-1,slide_002,1,Labyrinth,0.904704,0.109883,0.940563,0.945861,0.929959,0.971442,76,0.661686,0.394319,0.892934,0.931113
147,yolo26l-1280r640-1,slide_021,1,Labyrinth,0.682931,0.292507,0.732630,0.769860,0.683158,0.999580,77,0.647092,0.385388,0.653165,0.687337
148,yolo26l-1280r640-1,slide_021,2,Junctional Zone,0.824481,0.183739,0.895068,0.888796,0.868786,0.944723,60,0.406715,0.385388,0.653165,0.687337


In [17]:
def rank_models(df):
    """Crea ranking de modelos por diferentes criterios"""
    
    # Agrupar por modelo y calcular métricas globales
    model_summary = df.groupby('model').agg({
        'iou_mean': ['mean', 'std'],
        'dice_mean': 'mean',
        'precision_mean': 'mean',
        'recall_mean': 'mean',
        'pixel_accuracy': 'first',  # Es el mismo por imagen
        'n_tiles_evaluable': 'sum'
    }).round(4)
    
    # Aplanar columnas multiíndice
    model_summary.columns = ['_'.join(col).strip() for col in model_summary.columns]
    model_summary = model_summary.reset_index()
    
    # Calcular mIoU (promedio de IoU por clase, luego por modelo)
    miou_by_model = df.groupby(['model', 'class_name'])['iou_mean'].mean().groupby('model').mean()
    model_summary['miou'] = model_summary['model'].map(miou_by_model)
    
    # Score compuesto (puedes ajustar pesos)
    model_summary['score_compuesto'] = (
        0.4 * model_summary['iou_mean_mean'] +
        0.3 * model_summary['dice_mean_mean'] +
        0.2 * model_summary['recall_mean_mean'] +
        0.1 * model_summary['precision_mean_mean']
    )
    
    return model_summary.sort_values('score_compuesto', ascending=False)

ranking = rank_models(df)
print(ranking.to_string())

                                          model  iou_mean_mean  iou_mean_std  dice_mean_mean  precision_mean_mean  recall_mean_mean  pixel_accuracy_first  n_tiles_evaluable_sum      miou  score_compuesto
7                  yolo26x-256r256-ovlp50-100ep         0.8035        0.0977          0.8635               0.8679            0.9094                0.2581                   4823  0.803540          0.84912
3                       yolo26l-1280r640-ovl256         0.7909        0.1047          0.8637               0.8584            0.9093                0.2609                   1040  0.790860          0.84317
9                 yolo26x-768r512-ovlp150-100ep         0.7781        0.1121          0.8492               0.8575            0.8855                0.2534                   1497  0.778130          0.82885
2                            yolo26l-1280r640-1         0.7700        0.1155          0.8454               0.8458            0.8941                0.2549                   1040  0.7700

In [18]:
def analyze_by_class(df):
    """Comparación detallada por clase"""
    
    pivot_iou = df.pivot_table(
        index='model', 
        columns='class_name', 
        values='iou_mean', 
        aggfunc='mean'
    ).round(4)
    
    # Calcular ranking por clase
    rankings = {}
    for col in pivot_iou.columns:
        rankings[f'{col}_rank'] = pivot_iou[col].rank(ascending=False)
    
    rank_df = pd.DataFrame(rankings)
    pivot_iou = pd.concat([pivot_iou, rank_df], axis=1)
    
    return pivot_iou.sort_values('Decidua', ascending=False)

class_comparison = analyze_by_class(df)
print(class_comparison)

                                              Decidua  Junctional Zone  \
model                                                                    
yolo11x-1280r896-ovl256-w_bgs                  0.7573           0.7928   
yolo26x-256r256-ovlp50-100ep                   0.7398           0.7709   
yolo26l-1280r640-ovl256                        0.7346           0.7512   
yolo26x-512r256-ovlp150-100ep                  0.7181           0.7436   
yolo26l-1280r640-1                             0.7134           0.7559   
yolo26l-1024r512-100epochs-0902-resized512_1   0.7047           0.7769   
yolo26x-768r512-ovlp150-100ep                  0.6901           0.7566   
yolo26x-2048r1024-ovlp512-100ep                0.6547           0.6919   
yolo26x-256r128-ovlp50-100ep                   0.6526           0.7664   
yolo26l-1280r896-ovl256-w_bgs                  0.0000           0.0000   

                                              Labyrinth  Decidua_rank  \
model                                 

In [19]:
from scipy import stats

def statistical_comparison(df, metric='iou_mean'):
    """Compara si las diferencias entre modelos son significativas"""
    
    models = df['model'].unique()
    results = []
    
    # Test ANOVA para ver si hay diferencias significativas
    groups = [df[df['model'] == m][metric].values for m in models]
    f_stat, p_value = stats.f_oneway(*groups)
    
    print(f"ANOVA {metric}: F={f_stat:.4f}, p={p_value:.4f}")
    
    if p_value < 0.05:
        print("Diferencias significativas entre modelos detectadas")
        
        # Comparaciones por pares (t-test)
        for i, m1 in enumerate(models):
            for m2 in models[i+1:]:
                data1 = df[df['model'] == m1][metric]
                data2 = df[df['model'] == m2][metric]
                
                t_stat, p_val = stats.ttest_ind(data1, data2)
                
                results.append({
                    'modelo_1': m1,
                    'modelo_2': m2,
                    'mean_diff': data1.mean() - data2.mean(),
                    'p_value': p_val,
                    'significativo': p_val < 0.05
                })
    
    return pd.DataFrame(results)

sig_results = statistical_comparison(df)
print(sig_results[sig_results['significativo']].sort_values('mean_diff', ascending=False))

ANOVA iou_mean: F=71.3898, p=0.0000
Diferencias significativas entre modelos detectadas
                         modelo_1  \
1    yolo26x-256r256-ovlp50-100ep   
9    yolo26x-256r128-ovlp50-100ep   
19  yolo26l-1280r896-ovl256-w_bgs   
18  yolo26l-1280r896-ovl256-w_bgs   
20  yolo26l-1280r896-ovl256-w_bgs   
23  yolo26l-1280r896-ovl256-w_bgs   
22  yolo26l-1280r896-ovl256-w_bgs   
21  yolo26l-1280r896-ovl256-w_bgs   
17  yolo26l-1280r896-ovl256-w_bgs   

                                        modelo_2  mean_diff       p_value  \
1                  yolo26l-1280r896-ovl256-w_bgs   0.803540  1.555379e-23   
9                  yolo26l-1280r896-ovl256-w_bgs   0.766517  2.601105e-19   
19               yolo26x-2048r1024-ovlp512-100ep  -0.736915  1.067518e-15   
18                 yolo11x-1280r896-ovl256-w_bgs  -0.744105  5.121097e-22   
20  yolo26l-1024r512-100epochs-0902-resized512_1  -0.755392  1.114829e-22   
23                            yolo26l-1280r640-1  -0.770049  4.596664e-21   
22

In [20]:
df

,model,image,class_id,class_name,iou_mean,iou_std,iou_median,dice_mean,precision_mean,recall_mean,n_tiles_evaluable,coverage_mean,pixel_accuracy,pixel_accuracy_mean,pixel_accuracy_pred_based
0,yolo26x-256r256-ovlp50-100ep,slide_010,0,Decidua,0.839465,0.190772,0.903328,0.897461,0.921504,0.904443,157,0.619371,0.258108,0.899277,0.938311
1,yolo26x-256r256-ovlp50-100ep,slide_010,2,Junctional Zone,0.711795,0.240297,0.781030,0.803518,0.831153,0.861067,119,0.441911,0.258108,0.899277,0.938311
2,yolo26x-256r256-ovlp50-100ep,slide_010,1,Labyrinth,0.901462,0.166065,0.998123,0.938201,0.917137,0.979020,322,0.817772,0.258108,0.899277,0.938311
3,yolo26x-256r256-ovlp50-100ep,slide_056,0,Decidua,0.648589,0.340786,0.779502,0.720643,0.801006,0.727739,481,0.679333,0.369973,0.818347,0.850108
4,yolo26x-256r256-ovlp50-100ep,slide_056,2,Junctional Zone,0.680895,0.305065,0.771460,0.764161,0.768880,0.894095,483,0.709658,0.369973,0.818347,0.850108
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,yolo26l-1280r640-1,slide_002,2,Junctional Zone,0.788808,0.156917,0.820785,0.871894,0.857350,0.924284,62,0.429066,0.394319,0.892934,0.931113
146,yolo26l-1280r640-1,slide_002,1,Labyrinth,0.904704,0.109883,0.940563,0.945861,0.929959,0.971442,76,0.661686,0.394319,0.892934,0.931113
147,yolo26l-1280r640-1,slide_021,1,Labyrinth,0.682931,0.292507,0.732630,0.769860,0.683158,0.999580,77,0.647092,0.385388,0.653165,0.687337
148,yolo26l-1280r640-1,slide_021,2,Junctional Zone,0.824481,0.183739,0.895068,0.888796,0.868786,0.944723,60,0.406715,0.385388,0.653165,0.687337


In [21]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

def create_comparison_dashboard(df, output_path='comparison_dashboard.html'):
    """Dashboard comparativo de todos los modelos"""
    
    models = df['model'].unique()
    classes = df['class_name'].unique()
    
    # CORREGIDO: Usar paleta con más colores o generar colores dinámicamente
    # Opción 1: Usar una paleta más grande
    # colors = px.colors.qualitative.Bold + px.colors.qualitative.Vivid
    
    # Opción 2: Generar colores dinámicamente con una escala continua
    n_models = len(models)
    colors = px.colors.sample_colorscale(
        px.colors.sequential.Viridis, 
        [i/(n_models-1) if n_models > 1 else 0.5 for i in range(n_models)]
    )
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'mIoU por Modelo',
            'IoU por Clase y Modelo',
            'Precision vs Recall por Modelo',
            'Distribución de IoU (todos los tiles)'
        ),
        specs=[[{"type": "bar"}, {"type": "bar"}],
               [{"type": "scatter"}, {"type": "violin"}]]
    )
    
    # 1. mIoU global por modelo
    miou_data = df.groupby('model')['iou_mean'].mean().sort_values(ascending=False)
    fig.add_trace(
        go.Bar(x=miou_data.index, y=miou_data.values, 
               marker_color=colors, text=miou_data.values.round(3),
               textposition='auto', showlegend=False),
        row=1, col=1
    )
    
    # 2. IoU por clase
    class_colors = {'Decidua': '#e74c3c', 'Labyrinth': '#2ecc71', 'Junctional Zone': '#3498db'}
    for cls in classes:
        cls_data = df[df['class_name'] == cls].groupby('model')['iou_mean'].mean()
        fig.add_trace(
            go.Bar(name=cls, x=cls_data.index, y=cls_data.values,
                   marker_color=class_colors.get(cls, '#95a5a6')),
            row=1, col=2
        )
    
    # 3. Precision vs Recall - CORREGIDO: colors ahora tiene suficientes elementos
    for i, model in enumerate(models):
        model_data = df[df['model'] == model].groupby('class_name').agg({
            'precision_mean': 'mean',
            'recall_mean': 'mean'
        })
        fig.add_trace(
            go.Scatter(x=model_data['recall_mean'], 
                      y=model_data['precision_mean'],
                      mode='markers+text',
                      name=model,
                      text=model_data.index,
                      textposition='top center',
                      marker=dict(size=12, color=colors[i])),
            row=2, col=1
        )
    fig.update_xaxes(title_text='Recall', row=2, col=1)
    fig.update_yaxes(title_text='Precision', row=2, col=1)
    
    # 4. Distribución de IoU
    for i, model in enumerate(models):
        model_iou = df[df['model'] == model]['iou_mean']
        fig.add_trace(
            go.Violin(y=model_iou, name=model, 
                     fillcolor=colors[i], opacity=0.6,
                     box_visible=True, meanline_visible=True,
                     line_color='black', showlegend=False),
            row=2, col=2
        )
    
    # Layout global
    fig.update_layout(
        height=800, 
        width=1400, 
        title_text="Comparación de Modelos",
        template='plotly_white',
        barmode='group',
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
    )
    
    fig.update_yaxes(range=[0, 1], row=1, col=1)
    fig.update_yaxes(range=[0, 1], row=1, col=2)
    fig.update_xaxes(range=[0, 1], row=2, col=1)
    fig.update_yaxes(range=[0, 1], row=2, col=1)
    fig.update_yaxes(range=[0, 1], row=2, col=2)
    
    fig.write_html(output_path)
    print(f"Dashboard guardado: {output_path}")
    return fig

# Ejecutar
create_comparison_dashboard(df)

Dashboard guardado: comparison_dashboard.html


In [22]:
def generate_best_model_report(df, output_file='best_model_report.txt'):
    """Genera un reporte con el mejor modelo y justificación"""
    
    with open(output_file, 'w') as f:
        f.write("=" * 70 + "\n")
        f.write("ANÁLISIS COMPARATIVO DE MODELOS\n")
        f.write("=" * 70 + "\n\n")
        
        # Ranking general
        ranking = rank_models(df)
        f.write("RANKING GENERAL DE MODELOS:\n")
        f.write("-" * 50 + "\n")
        for idx, row in ranking.iterrows():
            f.write(f"{idx+1}. {row['model']}\n")
            f.write(f"   mIoU: {row['miou']:.4f}\n")
            f.write(f"   Score compuesto: {row['score_compuesto']:.4f}\n")
            f.write(f"   Pixel Accuracy: {row['pixel_accuracy_first']:.2%}\n\n")
        
        best_model = ranking.iloc[0]['model']
        f.write(f"\nMEJOR MODELO: {best_model}\n")
        f.write("=" * 70 + "\n\n")
        
        # Análisis por clase del mejor modelo
        best_data = df[df['model'] == best_model]
        f.write("Rendimiento por clase:\n")
        for cls in best_data['class_name'].unique():
            cls_data = best_data[best_data['class_name'] == cls]
            f.write(f"\n  {cls}:\n")
            f.write(f"    IoU: {cls_data['iou_mean'].mean():.4f} ± {cls_data['iou_mean'].std():.4f}\n")
            f.write(f"    Dice: {cls_data['dice_mean'].mean():.4f}\n")
            f.write(f"    Tiles evaluados: {cls_data['n_tiles_evaluable'].sum()}\n")
        
        # Comparación vs segundo mejor
        if len(ranking) > 1:
            second_model = ranking.iloc[1]['model']
            f.write(f"\n\nCOMPARACIÓN: {best_model} vs {second_model}\n")
            f.write("-" * 50 + "\n")
            
            best_miou = ranking.iloc[0]['miou']
            second_miou = ranking.iloc[1]['miou']
            improvement = ((best_miou - second_miou) / second_miou) * 100
            f.write(f"Mejora en mIoU: {improvement:.2f}%\n")
    
    print(f"Reporte guardado: {output_file}")

generate_best_model_report(df)

Reporte guardado: best_model_report.txt


## Análisis del mejor modelo

In [23]:
import json
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path
from collections import defaultdict
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from scipy import stats

# =========================
# CONFIGURACIÓN
# =========================

MODEL_NAME = "yolo26x-256r256-ovlp50-100ep"  # <-- Modifica con tu mejor modelo
BASE_DIR = f"/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/models/inference_1903/{MODEL_NAME}"
OUTPUT_ANALYSIS = os.path.join(BASE_DIR, "analysis_report")
os.makedirs(OUTPUT_ANALYSIS, exist_ok=True)

CLASS_NAMES = {0: "Decidua", 1: "Labyrinth", 2: "Junctional Zone"}
CLASS_COLORS = {'Decidua': '#e74c3c', 'Labyrinth': '#2ecc71', 'Junctional Zone': '#3498db'}

# =========================
# CARGA DE DATOS
# =========================

def load_model_data(base_dir):
    """Carga todos los JSON de un modelo específico"""
    json_files = glob.glob(os.path.join(base_dir, "**/stats_*.json"), recursive=True)
    
    records = []
    all_confusion_matrices = {}
    
    for json_path in json_files:
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        image_name = data.get('image', os.path.basename(json_path).replace('stats_', '').replace('.json', ''))
        
        # Guardar matriz de confusión
        if 'confusion_matrix' in data:
            all_confusion_matrices[image_name] = data['confusion_matrix']
        
        # Extraer métricas por clase
        for class_id, class_data in data.get('per_class', {}).items():
            record = {
                'image': image_name,
                'class_id': int(class_id),
                'class_name': class_data['class_name'],
                'iou': class_data['iou']['mean'],
                'iou_std': class_data['iou']['std'],
                'iou_median': class_data['iou']['median'],
                'iou_min': class_data['iou']['min'],
                'iou_max': class_data['iou']['max'],
                'iou_q25': class_data['iou']['q25'],
                'iou_q75': class_data['iou']['q75'],
                'dice': class_data['dice']['mean'],
                'dice_std': class_data['dice']['std'],
                'precision': class_data['precision']['mean'],
                'recall': class_data['recall']['mean'],
                'f1': class_data['dice']['mean'],  # F1 = Dice
                'n_tiles': class_data['n_tiles_evaluable'],
                'n_tiles_with_pred': class_data['n_tiles_with_pred'],
                'coverage_gt': class_data['coverage_mean'],
                'pixel_accuracy': data.get('confusion_matrix', {}).get('pixel_accuracy', 0),
                'pixel_accuracy_mean': data.get('confusion_matrix', {}).get('pixel_accuracy_mean', 0),
                'pixel_accuracy_pred_based': data.get('confusion_matrix', {}).get('pixel_accuracy_pred_based', 0)
            }
            records.append(record)
    
    return pd.DataFrame(records), all_confusion_matrices

df, confusion_matrices = load_model_data(BASE_DIR)
print(f"Datos cargados: {len(df)} registros de {df['image'].nunique()} imágenes")
print(f"Clases: {df['class_name'].unique()}")

# =========================
# ANÁLISIS ESTADÍSTICO
# =========================

def comprehensive_statistics(df):
    """Estadísticas completas por clase y globales"""
    
    stats_summary = []
    
    for cls in sorted(df['class_name'].unique()):
        cls_data = df[df['class_name'] == cls]
        
        stats = {
            'Clase': cls,
            'N_imágenes': cls_data['image'].nunique(),
            'N_tiles_total': cls_data['n_tiles'].sum(),
            'IoU_media': cls_data['iou'].mean(),
            'IoU_std': cls_data['iou'].std(),
            'IoU_mediana': cls_data['iou'].median(),
            'IoU_min': cls_data['iou'].min(),
            'IoU_max': cls_data['iou'].max(),
            'IoU_IQR': cls_data['iou_q75'].mean() - cls_data['iou_q25'].mean(),
            'Dice_media': cls_data['dice'].mean(),
            'Dice_std': cls_data['dice'].std(),
            'Precision': cls_data['precision'].mean(),
            'Recall': cls_data['recall'].mean(),
            'F1': cls_data['f1'].mean(),
            'Cobertura_GT_media': cls_data['coverage_gt'].mean(),
            'Tasa_predicción': (cls_data['n_tiles_with_pred'].sum() / cls_data['n_tiles'].sum()) * 100
        }
        stats_summary.append(stats)
    
    # Fila de promedios globales
    global_stats = {
        'Clase': 'GLOBAL',
        'N_imágenes': df['image'].nunique(),
        'N_tiles_total': df['n_tiles'].sum(),
        'IoU_media': df['iou'].mean(),
        'IoU_std': df['iou'].std(),
        'IoU_mediana': df['iou'].median(),
        'IoU_min': df['iou'].min(),
        'IoU_max': df['iou'].max(),
        'IoU_IQR': 0,
        'Dice_media': df['dice'].mean(),
        'Dice_std': df['dice'].std(),
        'Precision': df['precision'].mean(),
        'Recall': df['recall'].mean(),
        'F1': df['f1'].mean(),
        'Cobertura_GT_media': df['coverage_gt'].mean(),
        'Tasa_predicción': (df['n_tiles_with_pred'].sum() / df['n_tiles'].sum()) * 100
    }
    stats_summary.append(global_stats)
    
    return pd.DataFrame(stats_summary)

stats_df = comprehensive_statistics(df)
print("\n" + "="*80)
print("RESUMEN ESTADÍSTICO")
print("="*80)
print(stats_df.round(4).to_string(index=False))

# Guardar CSV
stats_df.to_csv(os.path.join(OUTPUT_ANALYSIS, 'statistics_summary.csv'), index=False)

# =========================
# ANÁLISIS POR IMAGEN
# =========================

def per_image_analysis(df):
    """Análisis detallado por imagen"""
    image_stats = df.groupby('image').agg({
        'iou': 'mean',
        'dice': 'mean',
        'precision': 'mean',
        'recall': 'mean',
        'pixel_accuracy': 'first',
        'n_tiles': 'sum'
    }).round(4)
    
    image_stats.columns = ['mIoU', 'mDice', 'mPrecision', 'mRecall', 'Pixel_Acc', 'Total_Tiles']
    image_stats = image_stats.sort_values('mIoU', ascending=False)
    
    return image_stats

image_df = per_image_analysis(df)
print("\n" + "="*80)
print("RANKING DE IMÁGENES POR mIoU")
print("="*80)
print(image_df.head(10).to_string())

image_df.to_csv(os.path.join(OUTPUT_ANALYSIS, 'per_image_metrics.csv'))

# =========================
# MATRIZ DE CONFUSIÓN PROMEDIO
# =========================

def aggregate_confusion_matrix(confusion_matrices):
    """Promedia todas las matrices de confusión"""
    if not confusion_matrices:
        return None
    
    matrices = [cm['matrix_norm'] for cm in confusion_matrices.values() if 'matrix_norm' in cm]
    if not matrices:
        return None
    
    avg_matrix = np.mean(matrices, axis=0)
    
    df_cm = pd.DataFrame(
        avg_matrix,
        index=[f'Pred: {CLASS_NAMES[i]}' for i in [0, 1, 2]],
        columns=[f'GT: {CLASS_NAMES[i]}' for i in [0, 1, 2]]
    )
    
    return avg_matrix, df_cm

avg_cm, df_cm = aggregate_confusion_matrix(confusion_matrices)
if avg_cm is not None:
    print("\n" + "="*80)
    print("MATRIZ DE CONFUSIÓN PROMEDIO (NORMALIZADA)")
    print("="*80)
    print(df_cm.round(3).to_string())

# =========================
# VISUALIZACIONES COMPLETAS
# =========================

def create_comprehensive_dashboard(df, stats_df, avg_cm, output_path):
    """Dashboard completo de análisis del modelo"""
    
    classes = sorted(df['class_name'].unique())
    n_classes = len(classes)
    
    # Crear figura con múltiples subplots
    fig = make_subplots(
        rows=3, cols=3,
        subplot_titles=(
            'IoU por Clase (Media ± Std)',
            'Métricas por Clase',
            'Distribución IoU (Violin)',
            'Precision vs Recall',
            'Matriz de Confusión',
            'IoU vs Cobertura GT',
            'Ranking de Imágenes',
            'Histograma IoU',
            'Boxplot por Clase'
        ),
        specs=[[{"type": "bar"}, {"type": "bar"}, {"type": "violin"}],
               [{"type": "scatter"}, {"type": "heatmap"}, {"type": "scatter"}],
               [{"type": "bar"}, {"type": "histogram"}, {"type": "box"}]],
        vertical_spacing=0.1,
        horizontal_spacing=0.08
    )
    
    # 1. IoU por clase con error bars
    for i, cls in enumerate(classes):
        cls_stats = stats_df[stats_df['Clase'] == cls].iloc[0]
        fig.add_trace(
            go.Bar(
                x=[cls], 
                y=[cls_stats['IoU_media']],
                error_y=dict(type='data', array=[cls_stats['IoU_std']], visible=True,
                           color='black', thickness=1.5, width=6),
                marker_color=CLASS_COLORS[cls],
                text=f"{cls_stats['IoU_media']:.3f}",
                textposition='outside',
                showlegend=False
            ),
            row=1, col=1
        )
    
    # Línea de referencia IoU=0.5
    fig.add_hline(y=0.5, line_dash="dash", line_color="gray", row=1, col=1)
    
    # 2. Métricas agrupadas (IoU, Dice, Precision, Recall)
    metrics = ['IoU_media', 'Dice_media', 'Precision', 'Recall']
    metric_labels = ['IoU', 'Dice', 'Precision', 'Recall']
    x_pos = np.arange(len(classes))
    width = 0.2
    
    for j, (metric, label) in enumerate(zip(metrics, metric_labels)):
        values = [stats_df[stats_df['Clase'] == cls][metric].values[0] for cls in classes]
        fig.add_trace(
            go.Bar(
                name=label,
                x=[x + (j-1.5)*width for x in x_pos],
                y=values,
                text=[f"{v:.2f}" for v in values],
                textposition='auto',
                width=width,
                showlegend=True
            ),
            row=1, col=2
        )
    
    fig.update_xaxes(tickvals=x_pos, ticktext=classes, row=1, col=2)
    
    # 3. Violin plot de distribución IoU por clase
    for cls in classes:
        cls_data = df[df['class_name'] == cls]['iou']
        fig.add_trace(
            go.Violin(
                y=cls_data,
                name=cls,
                fillcolor=CLASS_COLORS[cls],
                opacity=0.7,
                line_color='black',
                box_visible=True,
                meanline_visible=True,
                showlegend=False
            ),
            row=1, col=3
        )
    fig.update_yaxes(range=[0, 1], row=1, col=3)
    
    # 4. Precision vs Recall scatter
    for cls in classes:
        cls_data = df[df['class_name'] == cls]
        fig.add_trace(
            go.Scatter(
                x=cls_data['recall'],
                y=cls_data['precision'],
                mode='markers',
                name=cls,
                marker=dict(color=CLASS_COLORS[cls], size=8, opacity=0.6),
                text=cls_data['image'],
                hovertemplate='%{text}<br>Recall: %{x:.3f}<br>Precision: %{y:.3f}',
                showlegend=False
            ),
            row=2, col=1
        )
    fig.update_xaxes(range=[0, 1], title_text='Recall', row=2, col=1)
    fig.update_yaxes(range=[0, 1], title_text='Precision', row=2, col=1)
    
    # Línea de F1=0.5 (donde Precision=Recall)
    fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, 
                  line=dict(color="gray", dash="dash"),
                  row=2, col=1)
    
    # 5. Matriz de confusión heatmap
    if avg_cm is not None:
        fig.add_trace(
            go.Heatmap(
                z=avg_cm,
                x=[CLASS_NAMES[i] for i in [0, 1, 2]],
                y=[CLASS_NAMES[i] for i in [0, 1, 2]],
                colorscale='RdYlGn',
                zmin=0, zmax=1,
                text=avg_cm.round(2),
                texttemplate='%{text:.2f}',
                textfont=dict(size=14, color='black'),
                hovertemplate='Pred: %{y}<br>GT: %{x}<br>Valor: %{z:.3f}<extra></extra>',
                colorbar=dict(title='Valor', x=0.68, y=0.5, len=0.3),
                showscale=True
            ),
            row=2, col=2
        )
    
    # 6. IoU vs Cobertura GT
    for cls in classes:
        cls_data = df[df['class_name'] == cls]
        fig.add_trace(
            go.Scatter(
                x=cls_data['coverage_gt'],
                y=cls_data['iou'],
                mode='markers',
                name=cls,
                marker=dict(color=CLASS_COLORS[cls], size=8, opacity=0.6),
                text=cls_data['image'],
                hovertemplate='%{text}<br>Cobertura: %{x:.1%}<br>IoU: %{y:.3f}',
                showlegend=False
            ),
            row=2, col=3
        )
    fig.update_xaxes(title_text='Cobertura GT', row=2, col=3)
    fig.update_yaxes(title_text='IoU', range=[0, 1], row=2, col=3)
    
    # 7. Ranking de imágenes por mIoU
    image_ranking = df.groupby('image')['iou'].mean().sort_values(ascending=True)
    colors_ranking = ['#e74c3c' if v < 0.5 else '#f39c12' if v < 0.7 else '#2ecc71' 
                      for v in image_ranking.values]
    
    fig.add_trace(
        go.Bar(
            y=image_ranking.index,
            x=image_ranking.values,
            orientation='h',
            marker_color=colors_ranking,
            text=image_ranking.values.round(3),
            textposition='outside',
            showlegend=False
        ),
        row=3, col=1
    )
    fig.update_xaxes(range=[0, 1], title_text='mIoU', row=3, col=1)
    
    # 8. Histograma de IoU
    fig.add_trace(
        go.Histogram(
            x=df['iou'],
            nbinsx=20,
            marker_color='#3498db',
            opacity=0.7,
            showlegend=False,
            name='IoU'
        ),
        row=3, col=2
    )
    fig.add_vline(x=df['iou'].mean(), line_dash="dash", line_color="red", 
                  annotation_text=f"Media: {df['iou'].mean():.3f}", row=3, col=2)
    
    # 9. Boxplot por clase
    for cls in classes:
        cls_data = df[df['class_name'] == cls]['iou']
        fig.add_trace(
            go.Box(
                y=cls_data,
                name=cls,
                marker_color=CLASS_COLORS[cls],
                boxmean=True,
                showlegend=False
            ),
            row=3, col=3
        )
    fig.update_yaxes(range=[0, 1], title_text='IoU', row=3, col=3)
    
    # Layout general
    miou_global = df['iou'].mean()
    pixel_acc_global = df['pixel_accuracy'].mean() * 100
    
    fig.update_layout(
        title=dict(
            text=f'<b>Análisis Completo: {MODEL_NAME}</b><br>' +
                 f'mIoU Global: {miou_global:.4f} | Pixel Accuracy: {pixel_acc_global:.2f}% | ' +
                 f'Imágenes: {df["image"].nunique()} | Tiles: {df["n_tiles"].sum()}',
            font_size=16,
            x=0.5
        ),
        height=1200,
        width=1600,
        template='plotly_white',
        barmode='group',
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
    )
    
    fig.update_yaxes(range=[0, 1], row=1, col=1)
    fig.update_yaxes(range=[0, 1], row=1, col=2)
    
    fig.write_html(output_path)
    print(f"\nDashboard guardado: {output_path}")
    return fig

dashboard_path = os.path.join(OUTPUT_ANALYSIS, f'dashboard_{MODEL_NAME}.html')
fig = create_comprehensive_dashboard(df, stats_df, avg_cm, dashboard_path)

# =========================
# ANÁLISIS DE CORRELACIONES
# =========================

def correlation_analysis(df):
    """Análisis de correlaciones entre variables"""
    
    corr_vars = ['iou', 'dice', 'precision', 'recall', 'coverage_gt', 'n_tiles']
    corr_matrix = df[corr_vars].corr()
    
    fig_corr = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_vars,
        y=corr_vars,
        colorscale='RdBu',
        zmin=-1, zmax=1,
        text=corr_matrix.values.round(2),
        texttemplate='%{text:.2f}',
        textfont=dict(size=12),
        colorbar=dict(title='Correlación')
    ))
    
    fig_corr.update_layout(
        title=f'Matriz de Correlación - {MODEL_NAME}',
        height=600,
        width=600,
        template='plotly_white'
    )
    
    fig_corr.write_html(os.path.join(OUTPUT_ANALYSIS, 'correlation_matrix.html'))
    return corr_matrix

corr_matrix = correlation_analysis(df)
print("\n" + "="*80)
print("MATRIZ DE CORRELACIÓN")
print("="*80)
print(corr_matrix.round(3).to_string())

# =========================
# DETECCIÓN DE OUTLIERS
# =========================

def detect_outliers(df):
    """Detecta imágenes con rendimiento anómalo"""
    
    outliers = []
    
    for cls in df['class_name'].unique():
        cls_data = df[df['class_name'] == cls]
        Q1 = cls_data['iou'].quantile(0.25)
        Q3 = cls_data['iou'].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        cls_outliers = cls_data[(cls_data['iou'] < lower_bound) | (cls_data['iou'] > upper_bound)]
        
        for _, row in cls_outliers.iterrows():
            outliers.append({
                'image': row['image'],
                'class': cls,
                'iou': row['iou'],
                'type': 'Bajo' if row['iou'] < lower_bound else 'Alto',
                'threshold': lower_bound if row['iou'] < lower_bound else upper_bound
            })
    
    return pd.DataFrame(outliers)

outliers_df = detect_outliers(df)
if not outliers_df.empty:
    print("\n" + "="*80)
    print("OUTLIERS DETECTADOS (método IQR)")
    print("="*80)
    print(outliers_df.to_string(index=False))
    outliers_df.to_csv(os.path.join(OUTPUT_ANALYSIS, 'outliers.csv'), index=False)
else:
    print("\nNo se detectaron outliers significativos")

# =========================
# REPORTE FINAL
# =========================

def generate_final_report(df, stats_df, outliers_df):
    """Genera reporte de texto completo"""
    
    report_path = os.path.join(OUTPUT_ANALYSIS, f'REPORTE_{MODEL_NAME}.txt')
    
    with open(report_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write(f"ANÁLISIS COMPLETO DEL MODELO: {MODEL_NAME}\n")
        f.write("="*80 + "\n\n")
        
        # Resumen ejecutivo
        f.write("RESUMEN EJECUTIVO\n")
        f.write("-"*40 + "\n")
        f.write(f"Total de imágenes analizadas: {df['image'].nunique()}\n")
        f.write(f"Total de tiles evaluados: {df['n_tiles'].sum():,}\n")
        f.write(f"mIoU Global: {df['iou'].mean():.4f} ± {df['iou'].std():.4f}\n")
        f.write(f"Pixel Accuracy Global: {df['pixel_accuracy'].mean():.2%}\n")
        f.write(f"Rango de IoU: [{df['iou'].min():.3f}, {df['iou'].max():.3f}]\n\n")
        
        # Por clase
        f.write("RENDIMIENTO POR CLASE\n")
        f.write("-"*40 + "\n")
        for _, row in stats_df[stats_df['Clase'] != 'GLOBAL'].iterrows():
            f.write(f"\n{row['Clase']}:\n")
            f.write(f"  IoU: {row['IoU_media']:.4f} ± {row['IoU_std']:.4f}\n")
            f.write(f"  Dice/F1: {row['Dice_media']:.4f}\n")
            f.write(f"  Precision: {row['Precision']:.4f} | Recall: {row['Recall']:.4f}\n")
            f.write(f"  Cobertura GT: {row['Cobertura_GT_media']:.1%}\n")
            f.write(f"  Tasa de predicción: {row['Tasa_predicción']:.1f}%\n")
        
        # Ranking de imágenes
        f.write("\n\nTOP 5 IMÁGENES (mejor mIoU)\n")
        f.write("-"*40 + "\n")
        top5 = df.groupby('image')['iou'].mean().sort_values(ascending=False).head(5)
        for img, miou in top5.items():
            f.write(f"{img}: {miou:.4f}\n")
        
        f.write("\n\nBOTTOM 5 IMÁGENES (peor mIoU)\n")
        f.write("-"*40 + "\n")
        bottom5 = df.groupby('image')['iou'].mean().sort_values(ascending=True).head(5)
        for img, miou in bottom5.items():
            f.write(f"{img}: {miou:.4f}\n")
        
        # Outliers
        if not outliers_df.empty:
            f.write(f"\n\nOUTLIERS DETECTADOS: {len(outliers_df)}\n")
            f.write("-"*40 + "\n")
            for _, row in outliers_df.iterrows():
                f.write(f"{row['image']} - {row['class']}: IoU={row['iou']:.3f} ({row['type']})\n")
        
        # Conclusión
        f.write("\n\nCONCLUSIÓN\n")
        f.write("-"*40 + "\n")
        best_class = stats_df[stats_df['Clase'] != 'GLOBAL'].loc[
            stats_df[stats_df['Clase'] != 'GLOBAL']['IoU_media'].idxmax(), 'Clase']
        worst_class = stats_df[stats_df['Clase'] != 'GLOBAL'].loc[
            stats_df[stats_df['Clase'] != 'GLOBAL']['IoU_media'].idxmin(), 'Clase']
        
        f.write(f"Mejor clase: {best_class}\n")
        f.write(f"Clase con más dificultades: {worst_class}\n")
        
        if df['iou'].mean() > 0.7:
            f.write("El modelo muestra un rendimiento EXCELENTE (mIoU > 0.7)\n")
        elif df['iou'].mean() > 0.5:
            f.write("El modelo muestra un rendimiento ACEPTABLE (mIoU > 0.5)\n")
        else:
            f.write("El modelo muestra dificultades (mIoU < 0.5)\n")
    
    print(f"\nReporte final guardado: {report_path}")

generate_final_report(df, stats_df, outliers_df)

print("\n" + "="*80)
print("ANÁLISIS COMPLETADO")
print(f"Resultados guardados en: {OUTPUT_ANALYSIS}")
print("="*80)

Datos cargados: 15 registros de 5 imágenes
Clases: ['Decidua' 'Junctional Zone' 'Labyrinth']

RESUMEN ESTADÍSTICO
          Clase  N_imágenes  N_tiles_total  IoU_media  IoU_std  IoU_mediana  IoU_min  IoU_max  IoU_IQR  Dice_media  Dice_std  Precision  Recall     F1  Cobertura_GT_media  Tasa_predicción
        Decidua           5           1113     0.7398   0.1020       0.7552   0.6210   0.8395   0.3594      0.8108    0.0965     0.8452  0.8379 0.8108              0.5852          94.6990
Junctional Zone           5           1298     0.7709   0.0701       0.8035   0.6809   0.8398   0.3181      0.8434    0.0571     0.8406  0.9122 0.8434              0.5861          99.5378
      Labyrinth           5           2412     0.8999   0.0097       0.8953   0.8918   0.9160   0.1207      0.9363    0.0087     0.9177  0.9782 0.9363              0.8135         100.0000
         GLOBAL           5           4823     0.8035   0.0977       0.8347   0.6210   0.9160   0.0000      0.8635    0.0815     0.867

# Inference + postprocess

In [19]:


# =========================
# POST-PROCESAMIENTO
# =========================

def remove_small_components(geom, min_area=100):
    """
    Elimina componentes conectados con área menor al umbral.
    Input: shapely Geometry (Polygon o MultiPolygon)
    Output: shapely Geometry filtrado
    """
    if geom is None or geom.is_empty:
        return Polygon()
    
    polygons = extract_polygons(geom)
    filtered = [p for p in polygons if p.area >= min_area]
    
    if not filtered:
        return Polygon()
    if len(filtered) == 1:
        return filtered[0]
    return unary_union(filtered)


def apply_morphology_buffer(geom, operation='close', buffer_distance=2.0):
    """
    Operaciones morfológicas via buffer.
    'close' = dilate + erode (cierra huecos)
    'open' = erode + dilate (elimina ruido)
    """
    if geom is None or geom.is_empty:
        return Polygon()
    
    geom = make_valid(geom)
    
    if operation == 'close':
        # Dilatación seguida de erosión
        dilated = geom.buffer(buffer_distance, join_style=2)
        result = dilated.buffer(-buffer_distance, join_style=2)
    elif operation == 'open':
        # Erosión seguida de dilatación
        eroded = geom.buffer(-buffer_distance, join_style=2)
        # Si la erosión consume todo, devolver vacío
        if eroded.is_empty:
            return Polygon()
        result = eroded.buffer(buffer_distance, join_style=2)
    else:
        result = geom
    
    return make_valid(result)


def filter_by_confidence(pred_by_class_with_scores, class_thresholds=None):
    """
    Filtra predicciones por umbral de confianza por clase.
    Input: dict {class_id: [{'geometry': poly, 'score': float}, ...]}
    Output: dict {class_id: geometrías filtradas}
    """
    if class_thresholds is None:
        # Default: umbral más bajo para clase 0 (Decidua)
        class_thresholds = {0: 0.3, 1: 0.5, 2: 0.5}
    
    filtered_by_class = {}
    
    for cls_id, predictions in pred_by_class_with_scores.items():
        threshold = class_thresholds.get(cls_id, 0.5)
        
        # Filtrar por score
        valid_preds = [p for p in predictions if p['score'] >= threshold]
        
        if valid_preds:
            geometries = [p['geometry'] for p in valid_preds]
            filtered_by_class[cls_id] = unary_union(geometries)
        else:
            filtered_by_class[cls_id] = Polygon()
    
    return filtered_by_class


def apply_smoothing(geom, tolerance=1.0):
    """
    Suaviza bordes via simplify (Douglas-Peucker).
    tolerance: máxima distancia permitida entre geometría original y simplificada.
    """
    if geom is None or geom.is_empty:
        return Polygon()
    
    geom = make_valid(geom)
    smoothed = geom.simplify(tolerance, preserve_topology=True)
    return smoothed if not smoothed.is_empty else Polygon()


def spatial_consistency_filter(pred_by_class, window_size=50):
    """
    Filtro de coherencia espacial: remueve píxeles aislados de una clase
    rodeados por otra clase dominante.
    Simplificación: si un componente está completamente contenido en otro
    y es muy pequeño en comparación, se elimina o fusiona.
    """
    if not pred_by_class:
        return pred_by_class
    
    # Unión de todas las predicciones
    all_pred = unary_union([g for g in pred_by_class.values() if not g.is_empty])
    
    cleaned = {}
    for cls_id, geom in pred_by_class.items():
        if geom.is_empty:
            cleaned[cls_id] = geom
            continue
        
        # Para cada componente, verificar si está "aislado"
        polygons = extract_polygons(geom)
        valid_polys = []
        
        for poly in polygons:
            # Si el polígono es muy pequeño comparado con el total de su clase
            if poly.area < (geom.area * 0.01) and poly.area < 500:
                # Verificar si está cerca de otros de la misma clase
                buffer_zone = poly.buffer(window_size)
                nearby_same_class = geom.intersection(buffer_zone).area - poly.area
                
                if nearby_same_class < poly.area * 0.5:  # Aislado
                    continue  # Eliminar
            valid_polys.append(poly)
        
        if valid_polys:
            cleaned[cls_id] = unary_union(valid_polys)
        else:
            cleaned[cls_id] = Polygon()
    
    return cleaned


# =========================
# PIPELINE DE COMPARACIÓN
# =========================

def apply_postprocessing_pipeline(pred_by_class_with_scores, techniques_config):
    """
    Aplica pipeline de post-procesamiento según configuración.
    
    techniques_config: dict con:
        - 'confidence_thresholds': dict {class_id: threshold}
        - 'remove_small': min_area (int) o None
        - 'morphology': {'operation': 'close'/'open', 'distance': float} o None
        - 'smoothing': tolerance (float) o None
        - 'spatial_filter': window_size (int) o None
    """
    # Paso 1: Filtrar por confianza
    if 'confidence_thresholds' in techniques_config:
        pred_by_class = filter_by_confidence(
            pred_by_class_with_scores, 
            techniques_config['confidence_thresholds']
        )
    else:
        # Sin filtro de confianza, unir todo
        pred_by_class = {
            cls: unary_union([p['geometry'] for p in preds])
            for cls, preds in pred_by_class_with_scores.items()
        }
    
    # Paso 2: Remover componentes pequeños
    if 'remove_small' in techniques_config:
        min_area = techniques_config['remove_small']
        pred_by_class = {
            cls: remove_small_components(geom, min_area)
            for cls, geom in pred_by_class.items()
        }
    
    # Paso 3: Morfología
    if 'morphology' in techniques_config:
        morph = techniques_config['morphology']
        pred_by_class = {
            cls: apply_morphology_buffer(
                geom, 
                morph.get('operation', 'close'),
                morph.get('distance', 2.0)
            )
            for cls, geom in pred_by_class.items()
        }
    
    # Paso 4: Suavizado
    if 'smoothing' in techniques_config:
        tolerance = techniques_config['smoothing']
        pred_by_class = {
            cls: apply_smoothing(geom, tolerance)
            for cls, geom in pred_by_class.items()
        }
    
    # Paso 5: Filtro espacial
    if 'spatial_filter' in techniques_config:
        window = techniques_config['spatial_filter']
        pred_by_class = spatial_consistency_filter(pred_by_class, window)
    
    return pred_by_class


def compare_postprocessing_techniques(pred_by_class_with_scores, gt_by_class, 
                                      image_path, output_dir, image_name,
                                      class_names=None, colors_rgb=None):
    """
    Genera comparación visual y tabla de métricas para diferentes técnicas.
    """
    if class_names is None:
        class_names = {0: "Decidua", 1: "Labyrinth", 2: "Junctional Zone"}
    if colors_rgb is None:
        colors_rgb = {0: (231, 76, 60), 1: (46, 204, 113), 2: (52, 152, 219)}
    
    # Definir configuraciones a probar
    configs = {
        'Original (sin post-proc)': {},
        'Confianza adaptativa': {
            'confidence_thresholds': {0: 0.35, 1: 0.5, 2: 0.5}
        },
        'Remover pequeños (area<100)': {
            'remove_small': 200
        },
        'Cierre morfológico': {
            'morphology': {'operation': 'close', 'distance': 2.0}
        },
        'Apertura morfológica': {
            'morphology': {'operation': 'open', 'distance': 2.0}
        },
        'Suavizado': {
            'smoothing': 1.5
        },
        'Combinado (conf + small + close)': {
            'confidence_thresholds': {0: 0.35, 1: 0.5, 2: 0.5},
            'remove_small': 100,
            'morphology': {'operation': 'close', 'distance': 2.0}
        }
    }
    
    results = {}
    
    # Aplicar cada configuración
    for name, config in configs.items():
        pred_processed = apply_postprocessing_pipeline(pred_by_class_with_scores, config)
        
        # Calcular métricas globales
        metrics_by_class = {}
        for cls in sorted(gt_by_class.keys()):
            pred_geom = pred_processed.get(cls, Polygon())
            gt_geom = gt_by_class.get(cls, Polygon())
            metrics = compute_metrics(pred_geom, gt_geom)
            metrics_by_class[cls] = metrics
        
        results[name] = {
            'predictions': pred_processed,
            'metrics': metrics_by_class
        }
    
    # Crear visualización comparativa
    fig = create_comparison_plot(results, image_path, class_names, colors_rgb)
    
    # Guardar figura
    comparison_path = f"{output_dir}/comparison_{image_name}.png"
    fig.savefig(comparison_path, dpi=150, bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    plt.close(fig)
    print(f"   📊 Comparación guardada: {comparison_path}")
    
    # Crear tabla de métricas
    metrics_df = create_metrics_table(results, class_names)
    csv_path = f"{output_dir}/metrics_comparison_{image_name}.csv"
    metrics_df.to_csv(csv_path, index=False)
    print(f"   📈 Tabla de métricas: {csv_path}")
    
    # Imprimir resumen en consola
    print(f"\n{'='*60}")
    print("COMPARACIÓN DE TÉCNICAS")
    print(f"{'='*60}")
    print(metrics_df.to_string())
    
    return results, metrics_df


def create_comparison_plot(results, image_path, class_names, colors_rgb, max_size=1024):
    """
    Crea figura comparativa con todas las técnicas.
    """
    n_techniques = len(results)
    n_cols = 3
    n_rows = (n_techniques + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    axes = axes.flatten() if n_techniques > 1 else [axes]
    
    # Cargar imagen base
    img = Image.open(image_path).convert('RGBA')
    scale = 1.0
    if max(img.size) > max_size:
        scale = max_size / max(img.size)
        img = img.resize((int(img.size[0]*scale), int(img.size[1]*scale)), 
                        Image.Resampling.LANCZOS)
    
    for idx, (name, data) in enumerate(results.items()):
        ax = axes[idx]
        
        # Crear overlay
        overlay_img = create_overlay_for_comparison(
            img.copy(), data['predictions'], colors_rgb, scale
        )
        
        ax.imshow(overlay_img)
        ax.set_title(name, fontsize=12, fontweight='bold', pad=10)
        ax.axis('off')
        
        # Añadir métricas como texto en la imagen
        metrics_text = []
        for cls in sorted(class_names.keys()):
            if cls in data['metrics']:
                dice = data['metrics'][cls]['dice']
                iou = data['metrics'][cls]['iou']
                metrics_text.append(f"{class_names[cls][:4]}: D={dice:.3f}, I={iou:.3f}")
        
        ax.text(0.02, 0.98, '\n'.join(metrics_text), 
                transform=ax.transAxes, fontsize=8, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Ocultar ejes sobrantes
    for idx in range(len(results), len(axes)):
        axes[idx].axis('off')
    
    # Leyenda general
    legend_elements = [
        plt.Rectangle((0,0),1,1, facecolor=np.array(colors_rgb[i])/255, 
                     label=class_names[i], alpha=0.6)
        for i in sorted(class_names.keys())
    ]
    fig.legend(handles=legend_elements, loc='lower center', 
              ncol=len(class_names), bbox_to_anchor=(0.5, -0.02))
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.08)
    
    return fig


def create_overlay_for_comparison(base_img, pred_by_class, colors_rgb, scale):
    """
    Crea imagen con overlay de predicciones.
    """
    overlay = Image.new('RGBA', base_img.size, (0,0,0,0))
    draw = ImageDraw.Draw(overlay)
    
    for cls, geom in pred_by_class.items():
        if geom.is_empty:
            continue
        
        color = colors_rgb.get(cls, (128,128,128)) + (153,)  # 60% alpha
        
        polys = extract_polygons(geom)
        for poly in polys:
            coords = [(x*scale, y*scale) for x,y in poly.exterior.coords]
            if len(coords) >= 3:
                draw.polygon(coords, fill=color, outline=color[:3]+(255,))
    
    result = Image.alpha_composite(base_img, overlay)
    return result.convert('RGB')


def create_metrics_table(results, class_names):
    """
    Crea DataFrame comparativo de métricas.
    """
    rows = []
    
    for technique_name, data in results.items():
        row = {'Técnica': technique_name}
        
        # Métricas promedio
        all_dice = []
        all_iou = []
        all_precision = []
        all_recall = []
        
        for cls in sorted(class_names.keys()):
            if cls in data['metrics']:
                m = data['metrics'][cls]
                cls_name = class_names[cls]
                
                row[f'{cls_name}_Dice'] = round(m['dice'], 4)
                row[f'{cls_name}_IoU'] = round(m['iou'], 4)
                row[f'{cls_name}_Prec'] = round(m['precision'], 4)
                row[f'{cls_name}_Rec'] = round(m['recall'], 4)
                
                all_dice.append(m['dice'])
                all_iou.append(m['iou'])
                all_precision.append(m['precision'])
                all_recall.append(m['recall'])
        
        # Promedios
        row['mDice'] = round(np.mean(all_dice), 4) if all_dice else 0
        row['mIoU'] = round(np.mean(all_iou), 4) if all_iou else 0
        row['mPrecision'] = round(np.mean(all_precision), 4) if all_precision else 0
        row['mRecall'] = round(np.mean(all_recall), 4) if all_recall else 0
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    # Reordenar columnas: técnicas primero, promedios, luego por clase
    cols = ['Técnica', 'mDice', 'mIoU', 'mPrecision', 'mRecall']
    for cls in sorted(class_names.keys()):
        name = class_names[cls]
        cols.extend([f'{name}_Dice', f'{name}_IoU', f'{name}_Prec', f'{name}_Rec'])
    
    return df[cols]



In [21]:
models_paths = glob('/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/notebooks/runs/segment/YOLO-lci1/*/weights/best.pt')
images_paths_test = glob('/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/wsi_splitted/test/*.tif')
for model_path in models_paths[:1]:

    model_name = model_path.split('/')[-3]

    RAW_OUTPUT = '/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/models/inference_2403/'
    OUTPUT_DIR = os.path.join(RAW_OUTPUT, model_name)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    slice_model = int(model_name.split('-')[1].split('r')[-1])
    slice_size = slice_model
    overlap_ratio = .25
    confidence_threshold = 0.10
    MIN_GT_COVERAGE = 0.05

    name_to_id = {"Decidual": 0, "Decidua": 0, "Labyrinth": 1, "Junctional Zone": 2}
    class_names = {0: "Decidua", 1: "Labyrinth", 2: "Junctional Zone"}
    colors_rgb = {0: (231, 76, 60), 1: (46, 204, 113), 2: (52, 152, 219)}
    colors_plotly = ['#e74c3c', '#2ecc71', '#3498db']

    print(f"Modelo: {model_name}")
    print(f"Slice size: {slice_size} | Overlap: {overlap_ratio*100:.0f}% | Conf. threshold: {confidence_threshold} | Min GT coverage: {MIN_GT_COVERAGE*100:.0f}%")



    # =========================
    # PIPELINE PRINCIPAL
    # =========================

    

    print("Cargando modelo...")
    model = UltralyticsDetectionModel(
        model_path=model_path,
        confidence_threshold=confidence_threshold,
        load_at_init=True
    )

    for image_path in images_paths_test[:1]:
        image_name = os.path.basename(image_path).split('.')[0]
        annot_path = f"/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/annotations_resized/{image_name}.geojson"
        OUTPUT_DIR_IMAGE = os.path.join(OUTPUT_DIR, image_name)
        os.makedirs(OUTPUT_DIR_IMAGE, exist_ok=True)
        
        print(f"\n{'='*60}")
        print(f"PROCESANDO: {image_name}")
        print(f"{'='*60}")
        
        if not os.path.exists(annot_path):
            print(f"⚠️  GT no encontrado, saltando...")
            continue
        
        # Cargar imagen
        img = Image.open(image_path)
        img_width, img_height = img.size
        print(f"Tamaño: {img_width}x{img_height}")
        
        # Crear grid
        tiles = create_tile_grid(img_width, img_height, slice_size, overlap_ratio)
        print(f"Tiles: {len(tiles)}")
        
        # Inferencia SAHI
        print("SAHI inference...")
        result = get_sliced_prediction(
            image_path, model,
            slice_height=slice_size, slice_width=slice_size,
            overlap_height_ratio=overlap_ratio, overlap_width_ratio=overlap_ratio
        )
        
        # Procesar predicciones
        pred_by_class_with_scores = {}
        for pred in result.object_prediction_list:
            if pred.mask and hasattr(pred.mask, "segmentation"):
                class_id = pred.category.id
                score = pred.score.value if hasattr(pred.score, 'value') else float(pred.score)
                
                for poly in pred.mask.segmentation:
                    if len(poly) >= 6:
                        raw_poly = segmentation_to_polygon(poly)
                        valid_geom = make_valid(raw_poly)
                        polygons = extract_polygons(valid_geom)
                        
                        for shapely_poly in polygons:
                            if shapely_poly.area > 0:
                                pred_by_class_with_scores.setdefault(class_id, []).append({
                                    'geometry': shapely_poly, 'score': score
                                })
        
        pred_by_class = {cls: unary_union([p['geometry'] for p in preds]) 
                        for cls, preds in pred_by_class_with_scores.items()}
        
        
        # Después de construir pred_by_class_with_scores, agregar:
        decidua_scores = [p['score'] for p in pred_by_class_with_scores.get(0, [])]
        print(f"Decidua: {len(decidua_scores)} predicciones")
        print(f"  Scores min/max/mean: {min(decidua_scores):.3f}/{max(decidua_scores):.3f}/{np.mean(decidua_scores):.3f}")
        print(f"  <0.15: {sum(1 for s in decidua_scores if s < 0.15)}")
        print(f"  <0.30: {sum(1 for s in decidua_scores if s < 0.30)}")
        print(f"  <0.50: {sum(1 for s in decidua_scores if s < 0.50)}")
        
        # Cargar GT
        gt_by_class = {}
        with open(annot_path) as f:
            data = json.load(f)
        
        for feature in data.get("features", []):
            geom = make_valid(shape(feature["geometry"]))
            class_name = feature["properties"].get("name")
            if class_name in name_to_id:
                cls_id = name_to_id[class_name]
                gt_by_class.setdefault(cls_id, []).append(geom)
        
        gt_by_class = {cls: unary_union(geoms) for cls, geoms in gt_by_class.items()}
        
        # Calcular matriz de confusión normalizada
        print("Calculando matriz de confusión...")
        confusion_result = compute_confusion_matrix_normalized(pred_by_class, gt_by_class,img_width, img_height)
        print(f"   Pixel Accuracy: {confusion_result['pixel_accuracy_standard']*100:.2f}%")
        print("   Matriz normalizada (Pred→GT):")
        print(confusion_result['dataframe'].round(3).to_string())
        
        # Métricas por tile
        print(f"Calculando métricas por tile...")
        tile_results = [compute_tile_metrics_detailed(pred_by_class, gt_by_class, t) for t in tiles]
        
        # Agregar estadísticas
        aggregated = aggregate_statistics_by_class(tile_results)
        
        # Mostrar resumen


        results_comparison, metrics_df = compare_postprocessing_techniques(
        pred_by_class_with_scores,
        gt_by_class,
        image_path,
        OUTPUT_DIR_IMAGE,
        image_name,
        class_names=class_names,
        colors_rgb=colors_rgb
        )
        print(f"\n{'='*60}")
        print("RESULTADOS")
        print(f"{'='*60}")
        
        for cls in sorted(aggregated.keys()):
            s = aggregated[cls]
            print(f"\n📊 {s['class_name']}")
            print(f"   Tiles evaluables: {s['n_tiles_evaluable']}")
            print(f"   IoU: {s['iou']['mean']:.4f} ± {s['iou']['std']:.4f}")
            print(f"   Dice: {s['dice']['mean']:.4f} ± {s['dice']['std']:.4f}")
            print(f"   Recall: {s['recall']['mean']:.4f} | Precision: {s['precision']['mean']:.4f}")
        
        # Tabla y CSV
        df = create_summary_table(aggregated)
        print(f"\nTABLA RESUMEN:")
        print(df.to_string())
        
        csv_path = os.path.join(OUTPUT_DIR_IMAGE, f"summary_{image_name}.csv")
        df.to_csv(csv_path, index=False)
        print(f"\n💾 CSV: {csv_path}")
        
        # JSON
        json_path = os.path.join(OUTPUT_DIR_IMAGE, f"stats_{image_name}.json")
        with open(json_path, 'w') as f:
            export_data = {
                'image': image_name,
                'dimensions': (img_width, img_height),
                'per_class': {k: {kk: vv for kk, vv in v.items() if kk != 'raw_data'} 
                            for k, v in aggregated.items()},
                'confusion_matrix': {
                    'matrix_norm': confusion_result['matrix_norm'].tolist(),
                    'matrix_raw': confusion_result['matrix_raw'].tolist(),
                    'pixel_accuracy_standard': float(confusion_result['pixel_accuracy_standard']),
                    'pixel_accuracy_mean': float(confusion_result['pixel_accuracy_mean']),              
                    'pixel_accuracy_pred_based': float(confusion_result['pixel_accuracy_pred_based']),   
                }
            }
            json.dump(export_data, f, indent=2)
        print(f"💾 JSON: {json_path}")
        
        # Dashboard
        print("Generando dashboard...")
        dashboard_files = create_complete_dashboard(
            aggregated_stats=aggregated,
            tile_results_list=tile_results,
            image_path=image_path,
            pred_by_class=pred_by_class,
            gt_by_class=gt_by_class,
            img_width=img_width,
            img_height=img_height,
            image_name=image_name,
            output_dir=OUTPUT_DIR_IMAGE,
            confusion_result=confusion_result,
            export_png=False
        )
        
        # Liberar memoria
        img.close()
        del result
        gc.collect()

    print(f"\n{'='*60}")
    print("PROCESO COMPLETADO")
    print(f"{'='*60}")

Modelo: yolo26x-256r256-ovlp50-100ep
Slice size: 256 | Overlap: 25% | Conf. threshold: 0.1 | Min GT coverage: 5%
Cargando modelo...

PROCESANDO: slide_021
Tamaño: 9406x5429
Tiles: 1296
SAHI inference...
Performing prediction on 1372 slices.
Decidua: 242 predicciones
  Scores min/max/mean: 0.101/0.947/0.553
  <0.15: 35
  <0.30: 75
  <0.50: 105
Calculando matriz de confusión...
   Pixel Accuracy: 38.60%
   Matriz normalizada (Pred→GT):
                       GT: Decidua  GT: Labyrinth  GT: Junctional Zone
Pred: Decidua                0.907          0.001                0.092
Pred: Labyrinth              0.001          0.967                0.031
Pred: Junctional Zone        0.092          0.077                0.831
Calculando métricas por tile...
   📊 Comparación guardada: /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/models/inference_2403/yolo26x-256r256-ovlp50-100ep/slide_021/comparison_slide_021.png
   📈 Tabla de métricas: /media/lci1/fac1e379-9201-42aa-80e8-b6775b0a